# **Exploración de subastas del BOE**

Este cuaderno usa el recolector `boe-subastas` como librería para buscar subastas en el [Portal de Subastas del BOE](https://subastas.boe.es). Para ejecutarlo tú mismo, desde la carpeta del proyecto y con el entorno activado:

```bash
pip install -e ".[notebook]"
jupyter notebook notebooks/explore_auctions.ipynb
```

## **Parámetros**

SE debe de cambiar aquí lo que se quieras explorar. Los códigos son los mismos que en el CLI (`boe-subastas search --help`).

In [1]:
PROVINCE = "Illes Balears"  # código INE ("07") o nombre oficial de la provincia
STATUS = None               # None = todos los estados; o "PU", "EJ", "SU", "CA", "PC", "FS"
ASSET_TYPE = None           # None = todos; "I" inmuebles, "V" vehículos, "M" otros bienes muebles
LIMIT = 12                  # número máximo de sobres (cada lote es un sobre)
USE_AUTH = False            # True para ver el importe de las pujas (sesión de usuario registrado)

## **Preparación**

Importamos el recolector y creamos la conexión con el portal. Si se activa `USE_AUTH`, el cuaderno reutiliza la sesión guardada por el CLI.

In [2]:
import json
import logging
from pathlib import Path

import pandas as pd
import requests

from boe_subastas import auth, models
from boe_subastas.client.collector import collect
from boe_subastas.client.connection import Connection
from boe_subastas.client.search import search

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(message)s")
pd.set_option("display.max_colwidth", 80)


def eur(value):
    """Presenta un importe en euros."""
    if isinstance(value, (int, float)):
        return f"{value:,.2f} €".replace(",", "X").replace(".", ",").replace("X", ".")
    return "—" if value is None else str(value)


def when(value):
    """Acorta una fecha ISO a «AAAA-MM-DD HH:MM»."""
    return "—" if value is None else str(value)[:16].replace("T", " ")


session = requests.Session()
if USE_AUTH:
    auth.prepare_authenticated_session(session)
connection = Connection(session=session, authenticated=USE_AUTH)

## **Búsqueda y recolección**

`search` recorre el listado del buscador con los filtros elegidos; `collect` descarga la ficha completa de cada subasta y produce un **sobre** por lote.

In [3]:
filters = {"province": models.province_code(PROVINCE)}
if STATUS:
    filters["status"] = STATUS
if ASSET_TYPE:
    filters["asset_type"] = ASSET_TYPE

envelopes = []
for item in search(connection, filters):
    for external_id, url, data in collect(connection, item["identificador"], status=item["estado"], case_number=item["expediente"]):
        envelopes.append(models.envelope(external_id, url, data))
    if len(envelopes) >= LIMIT:
        break
envelopes = envelopes[:LIMIT]

auctions = {e["data"]["identificador_subasta"] for e in envelopes}
print(f"{len(envelopes)} sobres de {len(auctions)} subastas con los filtros {filters}")

12 sobres de 11 subastas con los filtros {'province': '07'}


Una fila por sobre con lo esencial: estado, tipo de subasta, qué se vende y dónde, valor de subasta, tasación y la mejor puja (o el mensaje literal del portal cuando el importe no es visible).

In [4]:
def summary_row(envelope):
    data = envelope["data"]
    auction = data["subasta"]
    economics = data["lote"] or auction
    asset = (data["bienes"] or [{}])[0]
    return {
        "sobre": envelope["meta"]["external_id"],
        "subasta": data["identificador_subasta"],
        "estado": auction["estado"],
        "tipo de subasta": auction["tipo"],
        "bien": " · ".join(v for v in (asset.get("tipo"), asset.get("subtipo")) if v),
        "localidad": asset.get("localidad"),
        "descripcion": data["descripcion"],
        "valor_subasta": economics.get("valor_subasta"),
        "tasacion": economics.get("tasacion"),
        "puja_mas_alta": data["pujas"]["puja_mas_alta"],
        "pujas": data["pujas"]["mensaje"],
        "concluye": auction["fecha_conclusion"],
    }

In [5]:
df = pd.DataFrame([summary_row(e) for e in envelopes])
AMOUNTS = ["valor_subasta", "tasacion", "puja_mas_alta"]

view = df.drop(columns=["subasta"]).set_index("sobre")
view[AMOUNTS] = view[AMOUNTS].map(eur)
view["concluye"] = view["concluye"].map(when)
view["descripcion"] = view["descripcion"].map(lambda t: "—" if t is None else (t if len(t) <= 60 else t[:59] + "…"))
view

,estado,tipo de subasta,bien,localidad,descripcion,valor_subasta,tasacion,puja_mas_alta,pujas,concluye
sobre,,,,,,,,,,
SUB-JA-2026-263926,Próxima apertura,JUDICIAL EN VÍA DE APREMIO,Inmueble · Vivienda,SANT ANTONI DE PORTMANY,LOTE ÚNICO: VIVIENDA EN PLANTA BAJA DE SAN ANTONIO.,"163.830,00 €","0,00 €",—,NaN,nan
SUB-JA-2026-265137,Próxima apertura,JUDICIAL EN VÍA DE APREMIO,Inmueble · Vivienda,PALMA,UMERO 5 DE ORDEN. Piso segundo-segunda que tiene su acceso …,"335.666,10 €","335.666,10 €",—,NaN,nan
SUB-JA-2026-263956/L1,Celebrándose,JUDICIAL EN VÍA DE APREMIO,Inmueble · Garaje,CAPDEPERA,FINCA Nº 12.474 -DESCRIPCION: 1) URBANA.- DEPARTAMENTO NUME…,"11.610,00 €","0,00 €",—,NaN,2026-09-23 18:00
SUB-JA-2026-263956/L2,Celebrándose,JUDICIAL EN VÍA DE APREMIO,Inmueble · Garaje,CAPDEPERA,FINCA Nº 12.475 . DESCRIPCION: 2) URBANA.- DEPARTAMENTO NUM…,"11.610,00 €","0,00 €",—,NaN,2026-09-23 18:00
SUB-JA-2026-265107,Celebrándose,JUDICIAL EN VÍA DE APREMIO,Inmueble · Vivienda,CALA RATJADA,"Número finca registral: Finca 20.559 de Capdepera, Tomo 506…","303.360,00 €","0,00 €",—,La subasta no ha recibido pujas.,2026-09-23 18:00
SUB-AT-2026-26R0786001024,Celebrándose,AGENCIA TRIBUTARIA,Inmueble · Solar,MARIA DE LA SALUT,100% PLENO DOMINIO. SOLAR. TN/ JAIME I . 07519 - MARIA DE L…,"71.810,00 €","71.810,00 €",—,La subasta no ha recibido pujas.,2026-09-21 18:00
SUB-AT-2026-26R0786001028,Celebrándose,AGENCIA TRIBUTARIA,Inmueble · Vivienda,FORMENTERA,"50% PLENO DOMINIO. VIVIENDA. CL/ DISIDOR MACABICH, 34 B BJ …","283.493,52 €","290.292,50 €",—,La subasta no ha recibido pujas.,2026-09-21 18:00
SUB-AT-2026-26R0786001029,Celebrándose,AGENCIA TRIBUTARIA,Inmueble · Solar,EIVISSA,"100% pleno dominio de solar sito en Cl Ramón Muntaner, 96 I…","11.018,70 €","11.018,70 €",—,La subasta no ha recibido pujas.,2026-09-21 18:00
SUB-AT-2026-26R0786001033,Celebrándose,AGENCIA TRIBUTARIA,Inmueble · Finca rústica,ALGAIDA,"50% DOMINIO ÚTIL RUSTICA POLIG 3, PARCELA 131 . 07210 - ALG…","93.463,20 €","155.772,00 €",—,La subasta no ha recibido pujas.,2026-09-21 18:00


## **Recuentos**

Cuántas subastas hay en cada estado (una subasta con varios lotes cuenta una sola vez) y qué se vende. Los importes son numéricos en el DataFrame, así que se pueden agregar directamente.

In [6]:
by_status = df.groupby("estado")["subasta"].nunique().rename("subastas").to_frame()
by_status.loc["Total"] = df["subasta"].nunique()
by_status

,subastas
estado,
Celebrándose,9
Próxima apertura,2
Total,11


In [7]:
by_asset = df.groupby("bien")["valor_subasta"].agg(sobres="count", minimo="min", mediana="median", maximo="max")
by_asset[["minimo", "mediana", "maximo"]] = by_asset[["minimo", "mediana", "maximo"]].map(eur)
by_asset.sort_values("sobres", ascending=False)

,sobres,minimo,mediana,maximo
bien,,,,
Inmueble · Vivienda,6,"163.830,00 €","260.577,01 €","335.666,10 €"
Inmueble · Finca rústica,2,"34.927,84 €","64.195,52 €","93.463,20 €"
Inmueble · Garaje,2,"11.610,00 €","11.610,00 €","11.610,00 €"
Inmueble · Solar,2,"11.018,70 €","41.414,35 €","71.810,00 €"


## **Ficha de una subasta**

El primer sobre, pestaña a pestaña: la subasta madre, sus condiciones económicas, los bienes y las pujas. Cambia `envelope` para ver otro.

In [8]:
envelope = envelopes[0]
data = envelope["data"]
print(envelope["meta"]["external_id"], "·", envelope["meta"]["url"])

SUB-JA-2026-263926 · https://subastas.boe.es/detalleSubasta.php?idSub=SUB-JA-2026-263926


In [9]:
def vertical(mapping, title):
    """Presenta un bloque de la ficha como tabla de dos columnas (listas y bloques anidados aparte)."""
    rows = {k: (eur(v) if isinstance(v, float) else v) for k, v in mapping.items() if not isinstance(v, (list, dict))}
    return pd.Series(rows, name=title, dtype=object).to_frame()


vertical(data["subasta"], "subasta")

,subasta
identificador,SUB-JA-2026-263926
tipo,JUDICIAL EN VÍA DE APREMIO
estado,Próxima apertura
expediente,0290/24
cuenta_expediente,0418 0000 06 0290 24
fecha_inicio,None
fecha_conclusion,None
cantidad_reclamada,"19.597,65 €"
lotes,0
forma_adjudicacion,None


In [10]:
vertical(data["subasta"]["autoridad_gestora"] or {}, "autoridad gestora")

,autoridad gestora
codigo,0702642001
descripcion,Sección Civil TI Eivissa. Plz.n 1
direccion,AV SAN CRISTOFORO S/N 4 ; 07800 IBIZA
telefono,971315746
fax,971194917
correo_electronico,scej.eivissa@justicia.es
otros,None


In [ ]:
assets = pd.DataFrame(data["bienes"]).drop(columns=["imagenes", "documentos", "otros"], errors="ignore")
assets.T.rename(columns=lambda i: f"bien {i + 1}")

,bien 1
numero,1
tipo,Inmueble
subtipo,Vivienda
descripcion,LOTE ÚNICO: VIVIENDA EN PLANTA BAJA DE SAN ANTONIO.
idufir,None
referencia_catastral,3654002CD5135S0013UF
direccion,"AVENIDA DOCTOR FLEMING Nº20 PLANTA BAJA, PUERTA 109"
codigo_postal,07820
localidad,SANT ANTONI DE PORTMANY
provincia,Illes Balears
